## Load the language model; set the GPU

In [1]:
import os
from nnsight import LanguageModel

os.environ["CUDA_VISIBLE_DEVICES"] = "7"

model = LanguageModel(
    "Qwen/Qwen3-32B",
    attn_implementation="eager"
)
model.to_empty(device="cuda:0")
model.device

/disk/u/gio/.conda/envs/rle/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device(type='cuda', index=0)

## Load the problem

In [2]:
import json

dataset = []
with open('../data/exp_2_hsp_res.jsonl', 'r') as f:
    for line in f:
        dataset.append(json.loads(line))
        
PROBLEM_2 = dataset[1]

## Integrated Gradient Function

In [9]:
import torch
from tqdm import trange

def integrated_gradients(
    model: LanguageModel,
    input_text: str,
    target_token_id: int, # Which output token to attribute
    baseline_id: int | None = None,
    interpolation_steps: int = 50
):
    if baseline_id is None:
        baseline_id = model.tokenizer.pad_token_id or model.tokenizer.eos_token_id
        
    # Get the baseline embedding
    baseline_embed = model.model.embed_tokens.weight[baseline_id].detach()
    
    # Tokenize the full input
    token_ids = model.tokenizer.encode(input_text, add_special_tokens=False)
    tokens = model.tokenizer.tokenize(input_text)
    
    # Get all token embeddings: shape (seq_len, hidden_dim)
    token_embeds = model.model.embed_tokens.weight[token_ids].detach()
    
    # Baseline is repeated for each position: shape (seq_len, hidden_dim)
    baseline_embeds = baseline_embed.unsqueeze(0).expand_as(token_embeds)
    
    # Difference between input and baseline
    x_minus_baseline = token_embeds - baseline_embeds # (seq_len, hidden_dim)
    
    # Accumulate gradients across interpolation steps
    accumulated_grads = torch.zeros_like(token_embeds).to(device="cpu")
    print(accumulated_grads.device)
    
    for step in trange(1, interpolation_steps + 1):
        alpha = step / interpolation_steps
        interpolated_embeds = baseline_embeds + alpha * x_minus_baseline
        
        with model.trace(input_text):
            # Move to correct device and add batch dimension INSIDE trace
            interpolated_embeds_traced = interpolated_embeds.unsqueeze(0).to(model.device).requires_grad_(True)
            
            # Override the embedding output
            model.model.embed_tokens.output = interpolated_embeds_traced
            
            # Get logits
            logits = model.output[0]
            target_logit = logits[0, -1, target_token_id]
            
            # Compute gradients
            target_logit.backward()
            grad = interpolated_embeds_traced.grad.save()
            
        print(f"{grad.device=}")
        
        accumulated_grads += grad.squeeze(0)
        
    # Average the gradients
    avg_grads = accumulated_grads / interpolation_steps
    
    print(avg_grads.device)
    
    print(x_minus_baseline.device)
    
    x_minus_baseline = x_minus_baseline.to(device='cpu')
    
    print(x_minus_baseline.device)
    
    # Integrated gradients = (x - x') * avg_grads
    ig_attributions = x_minus_baseline * avg_grads # (seq_len, hidden_dim)
    
    # sum across hidden dimension to get per-token attributino
    token_attributions = ig_attributions.sum(dim=-1) # (seq_len,)
    
    return {
        'tokens': tokens,
        'token_ids': token_ids,
        "attributions": token_attributions,
        "attributions_full": ig_attributions,
    }

## IG Saving Function

In [4]:
def to_json_safe(obj):
    if torch.is_tensor(obj):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: to_json_safe(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [to_json_safe(x) for x in obj]
    else:
        return obj

## Helper Methods

In [5]:
def extract_cot_minus_answer(full_cot):
    # Calculate the index of the last character of "\\boxed{"
    index_of_boxed = full_cot.rfind("\\boxed{", )
    index_of_answer = index_of_boxed + +len("\\boxed{")
    return full_cot[:index_of_answer]

def get_target_token_id(full_cot, cot_minus_answer):
    """Get the id of the first token of the final answer."""
    token_ids = model.tokenizer.encode(full_cot, add_special_tokens=False)
    prefix_tokens = model.tokenizer.encode(cot_minus_answer, add_special_tokens=False)
    target_token_id = token_ids[len(prefix_tokens)]
    return target_token_id

## Sanity Check

In [7]:
answer = PROBLEM_2['answer']
print(answer)

full_cot = PROBLEM_2['full_cot']
cot_minus_answer = extract_cot_minus_answer(full_cot)
print(cot_minus_answer)

target_id = get_target_token_id(full_cot, cot_minus_answer)
print(target_id)
print(model.tokenizer.decode(target_id))

10
<think>
Okay, so I need to figure out how many continuous paths there are from point A to point B along the segments of this figure without revisiting any of the six labeled points. Let me start by visualizing the Asymptote figure they provided. 

First, there's a rectangle with vertices at (0,0), (3,0), (3,2), and (0,2). Then there are some additional lines: from (0,2) to (1,0), from (1,0) to (3,2), and from (0,2) to (1.5,3.5), which is point A. Also, there's a line from (1.5,3.5) to (3,2). The labeled points are A at the top, B at the bottom-left, C at the top-left, D at the top-right, E at the bottom-right, F at the middle-bottom. 

So, the figure is a combination of a rectangle with a diagonal-like structure inside and a peak at the top. Let me try to sketch this mentally. The main rectangle is 3 units wide and 2 units tall. Then from point C (0,2), there's a line going down to F (1,0), then up to D (3,2), and back to C. Also, from C, there's a line going up to A (1.5,3.5), and 

## Compute IG

In [11]:
result = integrated_gradients(
    model=model,
    input_text=cot_minus_answer,
    target_token_id=target_id,
    interpolation_steps=20
)

for token, attr in zip(result["tokens"], result["attributions"]):
    print(f"{token:>10}: {attr.item():+.4f}")

cpu


  0%|          | 0/20 [00:00<?, ?it/s]

  5%|▌         | 1/20 [19:21<6:07:57, 1161.99s/it]

grad.device=device(type='cpu')


 10%|█         | 2/20 [39:32<5:57:05, 1190.33s/it]

...

grad.device=device(type='cpu')


 65%|██████▌   | 13/20 [4:17:09<2:18:25, 1186.51s/it]

grad.device=device(type='cpu')


 65%|██████▌   | 13/20 [4:20:44<2:20:23, 1203.42s/it]


KeyboardInterrupt: 

## Save Result

In [ ]:
with open('../data/attributions/result2.1.json', 'w') as f:
    json.dump(to_json_safe(result), f, indent=2)